In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.optim as optim

# ── 1. Load MNIST ──
transform = transforms.ToTensor()
dataset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)  # bigger batch
print("Dataset ready! Images:", len(dataset))

# ── 2. Improved VAE ──
# CHANGE 1: Latent dim increased from 32 → 64
# CHANGE 2: Deeper encoder/decoder for better reconstruction
LATENT_DIM = 64  # was 32 — more information = better reconstruction

class ImprovedVAE(nn.Module):
    def __init__(self):
        super(ImprovedVAE, self).__init__()

        # Deeper encoder
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU()
        )
        self.mean_layer = nn.Linear(128, LATENT_DIM)
        self.var_layer  = nn.Linear(128, LATENT_DIM)

        # Deeper decoder (mirrors encoder)
        self.decoder = nn.Sequential(
            nn.Linear(LATENT_DIM, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 784),
            nn.Sigmoid()
        )

    def encode(self, x):
        x = self.encoder(x)
        return self.mean_layer(x), self.var_layer(x)

    def reparameterise(self, mean, var):
        epsilon = torch.randn_like(var)
        return mean + var * epsilon

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mean, var = self.encode(x)
        z = self.reparameterise(mean, var)
        reconstructed = self.decode(z)
        return reconstructed, mean, var

model = ImprovedVAE()
print("Improved model ready!")
print(f"Latent dimension: {LATENT_DIM} floats = {LATENT_DIM * 4} bytes")

# ── 3. Improved Loss Function ──
# CHANGE 3: BCE loss instead of MSE — produces sharper MNIST digits
# CHANGE 4: Beta weighting on KL loss — better balance
BETA = 0.5  # weight on KL loss (lower = sharper reconstructions)

def loss_function(reconstructed, original, mean, var):
    original_flat = original.view(original.size(0), -1)

    # BCE loss (better than MSE for binary-ish images like MNIST)
    reconstruction_loss = nn.functional.binary_cross_entropy(
        reconstructed, original_flat, reduction='sum'
    )

    # KL divergence with beta weighting
    kl_loss = -0.5 * torch.sum(1 + var - mean.pow(2) - var.exp())

    return reconstruction_loss + BETA * kl_loss

# ── 4. Train for more epochs ──
# CHANGE 5: 50 epochs instead of 10
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
losses = []
print("Training started... (50 epochs, ~5-8 mins in Colab)")

for epoch in range(50):  # was 10
    total_loss = 0
    for images, _ in dataloader:
        optimizer.zero_grad()
        reconstructed, mean, var = model(images)
        loss = loss_function(reconstructed, images, mean, var)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    avg = total_loss / len(dataloader)
    losses.append(avg)

    if (epoch + 1) % 5 == 0:  # print every 5 epochs
        print(f"Epoch {epoch+1}/50 | Loss: {avg:.0f}")

print("Training complete!")

# ── 5. Save model ──
torch.save(model.state_dict(), 'vae_model_improved.pth')
print("Model saved as vae_model_improved.pth")

from google.colab import files
files.download('vae_model_improved.pth')

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 609kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.60MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.2MB/s]


Dataset ready! Images: 60000
Improved model ready!
Latent dimension: 64 floats = 256 bytes
Training started... (50 epochs, ~5-8 mins in Colab)
Epoch 5/50 | Loss: 12142
Epoch 10/50 | Loss: 10406
Epoch 15/50 | Loss: 9531
Epoch 20/50 | Loss: 8989
Epoch 25/50 | Loss: 8594
Epoch 30/50 | Loss: 8454
Epoch 35/50 | Loss: 8334
Epoch 40/50 | Loss: 8223
Epoch 45/50 | Loss: 8074
Epoch 50/50 | Loss: 8025
Training complete!
Model saved as vae_model_improved.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>